# Analyze Business Cards with Content Understanding

Use this notebook to run the same flow as `read-card.py`, one step at a time. It submits a business card image to the analyzer created by `create-analyzer.py` or `create-analyzer.ipynb`, saves the full response to `results.json`, and prints the extracted fields.

## 1. Confirm Prerequisites

Before running this notebook, create the analyzer first. Confirm that `workshop/.env` includes `CONTENT_UNDERSTANDING_ENDPOINT`, `CONTENT_UNDERSTANDING_KEY` if you want key-based authentication, and `ANALYZER_NAME`. If the key is empty, run `az login` first.

In [1]:
from pathlib import Path
import json
import os

from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

## 2. Locate the Notebook Files

This cell resolves the notebook folder, the shared environment file, both sample business card images, and the output file.

In [2]:
notebook_dir = Path.cwd()
if notebook_dir.name != "02-create-analyzers-python-sdk":
    notebook_dir = Path("workshop/lab-02-content-understanding/02-create-analyzers-python-sdk").resolve()

env_path = notebook_dir.parents[1] / ".env"
first_card_path = notebook_dir / "biz-card-1.png"
second_card_path = notebook_dir / "biz-card-2.png"
output_path = notebook_dir / "results.json"

print(f"Notebook folder: {notebook_dir}")
print(f"Environment file: {env_path}")
print(f"First card: {first_card_path}")
print(f"Second card: {second_card_path}")
print(f"Results file: {output_path}")

Notebook folder: c:\Users\algut\repos\alesaez\content-processing-solution\workshop\lab-02-content-understanding\02-create-analyzers-python-sdk
Environment file: c:\Users\algut\repos\alesaez\content-processing-solution\workshop\.env
First card: c:\Users\algut\repos\alesaez\content-processing-solution\workshop\lab-02-content-understanding\02-create-analyzers-python-sdk\biz-card-1.png
Second card: c:\Users\algut\repos\alesaez\content-processing-solution\workshop\lab-02-content-understanding\02-create-analyzers-python-sdk\biz-card-2.png
Results file: c:\Users\algut\repos\alesaez\content-processing-solution\workshop\lab-02-content-understanding\02-create-analyzers-python-sdk\results.json


## 3. Define Configuration Helpers

These helpers match `read-card.py`: placeholder values are treated as empty, and `CONTENT_UNDERSTANDING_KEY` is optional.

In [3]:
def optional_setting(name: str) -> str:
    value = (os.getenv(name) or "").strip()
    return "" if not value or value.startswith("<") else value


def require_setting(name: str) -> str:
    value = optional_setting(name)
    if not value:
        raise ValueError(f"Set {name} in workshop/.env")
    return value


def get_credential():
    key = optional_setting("CONTENT_UNDERSTANDING_KEY")
    return AzureKeyCredential(key) if key else DefaultAzureCredential()

## 4. Load Environment Settings

Load the Content Understanding endpoint and analyzer name from the shared environment file.

In [4]:
load_dotenv(env_path, override=True)

ai_svc_endpoint = require_setting("CONTENT_UNDERSTANDING_ENDPOINT")
analyzer = require_setting("ANALYZER_NAME")
auth_mode = "AzureKeyCredential" if optional_setting("CONTENT_UNDERSTANDING_KEY") else "DefaultAzureCredential"

print(f"Endpoint: {ai_svc_endpoint}")
print(f"Analyzer name: {analyzer}")
print(f"Authentication mode: {auth_mode}")

Endpoint: https://4ublv6yuwkfh4-aifoundry.services.ai.azure.com/
Analyzer name: bizcardanalyzer
Authentication mode: DefaultAzureCredential


## 5. Choose a Business Card Image

Set `image_file` to either sample image. Re-run this cell with a different path when you want to analyze another card.

In [11]:
# image_file = first_card_path
image_file = second_card_path

if not image_file.exists():
    raise FileNotFoundError(f"Business card image not found: {image_file}")

print(f"Selected image: {image_file.name}")

Selected image: biz-card-2.png


## 6. Create the Content Understanding Client

Create the SDK client with the selected credential.

In [12]:
client = ContentUnderstandingClient(
    endpoint=ai_svc_endpoint,
    credential=get_credential()
)

client

## 7. Submit the Image for Analysis

Read the selected image bytes and submit them to the analyzer. The SDK waits for the long-running analysis operation to finish when `poller.result()` runs.

In [13]:
print(f"Analyzing {image_file.name}")
image_data = image_file.read_bytes()

print("Submitting request...")
poller = client.begin_analyze_binary(
    analyzer_id=analyzer,
    binary_input=image_data,
    content_type="image/png"
)

result = poller.result()
print("Analysis succeeded.")

Analyzing biz-card-2.png
Submitting request...
Analysis succeeded.


## 8. Save the Full JSON Response

Save the full response to `results.json` so you can inspect the raw service output.

In [14]:
with output_path.open("w", encoding="utf-8") as json_file:
    json.dump(dict(result), json_file, indent=4, default=str)

print(f"Response saved in {output_path}")

Response saved in c:\Users\algut\repos\alesaez\content-processing-solution\workshop\lab-02-content-understanding\02-create-analyzers-python-sdk\results.json


## 9. Review Extracted Fields

Print the extracted business card fields from the analyzer response.

In [15]:
extracted_fields = []

for content in result.contents:
    if hasattr(content, "fields") and content.fields:
        for field_name, field_data in content.fields.items():
            value = field_data.value if hasattr(field_data, "value") else None
            extracted_fields.append((field_name, value))

if not extracted_fields:
    print("No extracted fields were returned.")
else:
    for field_name, value in extracted_fields:
        print(f"{field_name}: {value}")

Company: Contoso Ltd.
Name: Marie Duartes
Title: Customer Advisor
Email: marie@contoso.com
Phone: 555-010-9876


## 10. Analyze the Second Card

To analyze the second sample, go back to the image selection cell, switch `image_file` to `second_card_path`, and run the remaining cells again.